# 13주차_scrap 02_2025195988_김라래

- 이름 / 학번: 2025195988_김라래
- 과제 내용: `02-2.ipynb` 파일을 수정하여 기존 쪽수 정보 외에 다른 정보를 추출하고 신규 데이터프레임 생성
- 선택한 추가 정보: **무게**
- 옵션 과제 반영: `get_page_cnt(isbn)` 함수 안에서 **쪽수와 무게를 함께 추출**하고, 두 개의 컬럼을 추가


## 1. 라이브러리 불러오기

기존 `02-2.ipynb`와 같이 `pandas`, `requests`, `BeautifulSoup`를 사용.


In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

print("라이브러리 불러오기 완료")

라이브러리 불러오기 완료


## 2. 도서 데이터 불러오기

02-2 교안과 같이, `gdown`으로 도서 데이터를 내려받아 사용.

In [2]:
import pandas as pd
import gdown

gdown.download('https://bit.ly/3q9SZix', '20s_best_book.json', quiet=True)

books_df = pd.read_json('20s_best_book.json')

print("도서 데이터 다운로드 성공")

books_df.head()

도서 데이터 다운로드 성공


,no,ranking,bookname,authors,publisher,publication_year,isbn13,addition_symbol,vol,class_no,class_nm,loan_count,bookImageURL
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,허블,2019,9791190090018,03810,,813.7,문학 > 한국문학 > 소설,461,https://image.aladin.co.kr/product/19359/16/co...
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,팩토리나인,2020,9791165341909,03810,,813.7,문학 > 한국문학 > 소설,387,https://image.aladin.co.kr/product/24512/70/co...
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,난다,2019,9791188862290,03810,,813.7,문학 > 한국문학 > 소설,383,https://image.aladin.co.kr/product/19804/82/co...
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,문학동네,2020,9788954672214,03810,,813.7,문학 > 한국문학 > 소설,370,https://image.aladin.co.kr/product/24131/37/co...
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,창비,2017,9788936434267,03810,,813.7,문학 > 한국문학 > 소설,365,http://image.aladin.co.kr/product/16839/4/cove...


## 3. 필요한 컬럼만 선택하기

도서 번호, 순위, 도서명, 저자, 출판사, 출판연도, ISBN 정보를 사용.


In [3]:
books = books_df[['no', 'ranking', 'bookname', 'authors', 'publisher',
                  'publication_year', 'isbn13']]

books.head()

,no,ranking,bookname,authors,publisher,publication_year,isbn13
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,허블,2019,9791190090018
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,팩토리나인,2020,9791165341909
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,난다,2019,9791188862290
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,문학동네,2020,9788954672214
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,창비,2017,9788936434267


## 4. 검색 결과 페이지 가져오기

기존 코드에서는 ISBN으로 YES24 검색 페이지를 가져옴.

In [4]:
isbn = 9791190090018
url = 'http://www.yes24.com/Product/Search?domain=BOOK&query={}'

try:
    r = requests.get(url.format(isbn), timeout=5)
    search_html = r.text
    print("검색 페이지 가져오기 성공")
except Exception as e:
    search_html = '''
    <html>
    <body>
        <a class="gd_name" href="/Product/Goods/001">우리가 빛의 속도로 갈 수 없다면</a>
    </body>
    </html>
    '''
    print("검색 페이지 가져오기 실패: 예시 HTML로 진행합니다.")

print(search_html[:300])

검색 페이지 가져오기 성공





    
	<!DOCTYPE html >
	<html lang="ko">

<head>
	<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
	<meta http-equiv="Content-Type" content="text/html;charset=utf-8" />
	<meta http-equiv="Accept-CH" content="dpr, width, viewport-width, rtt, downlink, ect, UA, UA-Platform, UA-


## 5. BeautifulSoup으로 상품 링크 찾기

`BeautifulSoup`으로 HTML을 파싱하고, `class`가 `gd_name`인 링크를 찾음.


In [5]:
soup = BeautifulSoup(search_html, 'html.parser')
prd_link = soup.find('a', attrs={'class': 'gd_name'})

print(prd_link)
print(prd_link['href'])

<a class="gd_name" href="/product/goods/74261416" onclick="wiseLogV2('S', '101_005_003_001', ''); setGoodsClickExtraCodeHub('032', '9791190090018', '74261416', '0',this);">우리가 빛의 속도로 갈 수 없다면</a>
/product/goods/74261416


## 6. 상세 페이지에서 쪽수, 무게, 크기 정보 확인

`infoset_specific` 영역에서 `쪽수, 무게, 크기` 항목을 찾고, 쪽수와 무게 추출.


In [6]:
# 상세 페이지 가져오기
try:
    detail_url = 'http://www.yes24.com' + prd_link['href']
    r = requests.get(detail_url, timeout=5)
    detail_html = r.text
    print("상세 페이지 가져오기 성공")
except Exception as e:
    detail_html = '''
    <html>
    <body>
      <div id="infoset_specific">
        <table>
          <tr>
            <th>쪽수, 무게, 크기</th>
            <td>330쪽 | 496g | 130*205*30mm</td>
          </tr>
        </table>
      </div>
    </body>
    </html>
    '''
    print("상세 페이지 가져오기 실패: 예시 HTML로 진행합니다.")

soup = BeautifulSoup(detail_html, 'html.parser')
prd_detail = soup.find('div', attrs={'id': 'infoset_specific'})
print(prd_detail)

상세 페이지 가져오기 성공
<div class="gd_infoSet infoSet_noLine" id="infoset_specific">
<div class="tm_infoSet">
<h4 class="tit_txt">품목정보</h4>
</div>
<div class="infoSetCont_wrap">
<div class="yesTb">
<table class="tb_nor tb_vertical" summary="품목정보 국내도서, 외국도서 " width="100%">
<caption>품목정보</caption>
<colgroup>
<col width="170"/>
<col width="*"/>
</colgroup>
<tbody class="b_size">
<tr>
<th class="txt" scope="row">발행일</th>
<td class="txt lastCol">2019년 06월 24일</td>
</tr>
<tr>
<th class="txt" scope="row">쪽수, 무게, 크기</th>
<td class="txt lastCol">344쪽 | 496g | 130*198*30mm</td>
</tr>
<tr>
<th class="txt" scope="row">ISBN13</th>
<td class="txt lastCol">9791190090018</td>
</tr>
<tr>
<th class="txt" scope="row">ISBN10</th>
<td class="txt lastCol">1190090015</td>
</tr>
</tbody>
</table>
</div>
</div>
<script type="text/javascript">
        if ($("#infoset_specific table tbody tr").length == 0) {
            $("#infoset_specific").remove();
        }
    </script>
</div>


## 7. 한 권의 도서에서 쪽수와 무게 추출하기

`쪽수, 무게, 크기` 행을 찾은 뒤, `|` 기준으로 문자열을 나누어 쪽수와 무게를 가져옴.


In [7]:
prd_tr_list = prd_detail.find_all('tr')

page_info = ''
weight_info = ''

for tr in prd_tr_list:
    if tr.find('th').get_text().strip() == '쪽수, 무게, 크기':
        info_text = tr.find('td').get_text().strip()
        info_list = [x.strip() for x in info_text.split('|')]
        page_info = info_list[0]
        weight_info = info_list[1]
        break

print("원본 정보:", info_text)
print("쪽수:", page_info)
print("무게:", weight_info)

원본 정보: 344쪽 | 496g | 130*198*30mm
쪽수: 344쪽
무게: 496g


## 8. 쪽수와 무게를 함께 가져오는 함수 만들기

기존 함수 이름인 `get_page_cnt(isbn)`을 유지하되, 반환값을 쪽수와 무게 두 개로 구성함.  
요청 실패 또는 정보가 없을 경우 빈 문자열을 반환하도록 처리.


In [8]:
def get_page_cnt(isbn):
    # YES24 도서 검색 페이지 URL
    url = 'http://www.yes24.com/Product/Search?domain=BOOK&query={}'

    # ISBN으로 검색 페이지 가져오기
    r = requests.get(url.format(isbn))
    soup = BeautifulSoup(r.text, 'html.parser')

    # 검색 결과에서 도서명 링크 찾기
    prd_info = soup.find('a', attrs={'class': 'gd_name'})

    # 검색 결과가 없으면 빈 값 반환
    if prd_info is None:
        return '', ''

    # 상세 페이지 가져오기
    detail_url = 'http://www.yes24.com' + prd_info['href']
    r = requests.get(detail_url)
    soup = BeautifulSoup(r.text, 'html.parser')

    # 상세 정보 영역 찾기
    prd_detail = soup.find('div', attrs={'id': 'infoset_specific'})
    if prd_detail is None:
        return '', ''

    # 쪽수, 무게, 크기 항목 찾기
    prd_tr_list = prd_detail.find_all('tr')

    for tr in prd_tr_list:
        th = tr.find('th')
        td = tr.find('td')

        if th is not None and td is not None:
            if th.get_text().strip() == '쪽수, 무게, 크기':
                info_text = td.get_text().strip()
                info_list = [x.strip() for x in info_text.split('|')]

                page_cnt = info_list[0] if len(info_list) > 0 else ''
                weight = info_list[1] if len(info_list) > 1 else ''

                return page_cnt, weight

    return '', ''

## 9. 함수 실행 확인

대표 ISBN 하나를 넣어 함수가 쪽수와 무게를 함께 반환하는지 확인.


In [9]:
get_page_cnt(9791190090018)

('344쪽', '496g')

## 10. 상위 10개 도서에 적용하기

기존 노트북과 같이 `head(10)`으로 일부 도서를 선택한 뒤, `apply()`를 사용해 ISBN별 정보를 추출한다.


In [10]:
top10_books = books.head(10).copy()
top10_books

,no,ranking,bookname,authors,publisher,publication_year,isbn13
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,허블,2019,9791190090018
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,팩토리나인,2020,9791165341909
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,난다,2019,9791188862290
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,문학동네,2020,9788954672214
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,창비,2017,9788936434267
5,6,6,피프티 피플 :정세랑 장편소설,지은이: 정세랑,창비,2016,9788936434243
6,7,7,목소리를 드릴게요 :정세랑 소설집,지은이: 정세랑,아작,2020,9791165300005
7,8,8,나미야 잡화점의 기적 :히가시노 게이고 장편소설,지은이: 히가시노 게이고 ;옮긴이: 양윤옥,현대문학,2012,9788972756194
8,9,9,선량한 차별주의자,김지혜 지음,창비,2019,9788936477196
9,10,9,쇼코의 미소 :최은영 소설,지은이: 최은영,문학동네,2016,9788954641630


In [11]:
def get_page_weight(row):
    isbn = row['isbn13']
    return get_page_cnt(isbn)

page_weight = top10_books.apply(get_page_weight, axis=1)
print(page_weight)

0    (344쪽, 496g)
1    (300쪽, 358g)
2    (228쪽, 296g)
3    (340쪽, 412g)
4    (264쪽, 388g)
5    (396쪽, 512g)
6    (272쪽, 318g)
7            (, )
8    (244쪽, 324g)
9    (296쪽, 406g)
dtype: object


## 11. 쪽수와 무게 컬럼 만들기

함수의 반환값을 데이터프레임으로 바꾸고 컬럼명 지정.


In [12]:
page_weight_df = pd.DataFrame(page_weight.tolist(),
                              columns=['page_count', 'weight'],
                              index=top10_books.index)

page_weight_df

,page_count,weight
0,344쪽,496g
1,300쪽,358g
2,228쪽,296g
3,340쪽,412g
4,264쪽,388g
5,396쪽,512g
6,272쪽,318g
7,,
8,244쪽,324g
9,296쪽,406g


## 12. 신규 데이터프레임 생성

기존 도서 데이터에 `page_count`, `weight` 컬럼을 붙여 새로운 데이터프레임을 만든다.


In [13]:
top10_with_page_weight = pd.merge(top10_books, page_weight_df,
                                    left_index=True, right_index=True)

top10_with_page_weight

,no,ranking,bookname,authors,publisher,publication_year,isbn13,page_count,weight
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,허블,2019,9791190090018,344쪽,496g
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,팩토리나인,2020,9791165341909,300쪽,358g
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,난다,2019,9791188862290,228쪽,296g
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,문학동네,2020,9788954672214,340쪽,412g
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,창비,2017,9788936434267,264쪽,388g
5,6,6,피프티 피플 :정세랑 장편소설,지은이: 정세랑,창비,2016,9788936434243,396쪽,512g
6,7,7,목소리를 드릴게요 :정세랑 소설집,지은이: 정세랑,아작,2020,9791165300005,272쪽,318g
7,8,8,나미야 잡화점의 기적 :히가시노 게이고 장편소설,지은이: 히가시노 게이고 ;옮긴이: 양윤옥,현대문학,2012,9788972756194,,
8,9,9,선량한 차별주의자,김지혜 지음,창비,2019,9788936477196,244쪽,324g
9,10,9,쇼코의 미소 :최은영 소설,지은이: 최은영,문학동네,2016,9788954641630,296쪽,406g


## 13. 무게 정보만 포함한 데이터프레임도 확인

1번 요구사항에 맞춰 새롭게 추출한 정보인 `weight`를 포함하는 신규 데이터프레임을 별도로 확인한다.


In [14]:
top10_with_weight = top10_with_page_weight[['no', 'ranking', 'bookname',
                                            'authors', 'isbn13', 'weight']]

top10_with_weight

,no,ranking,bookname,authors,isbn13,weight
0,1,1,우리가 빛의 속도로 갈 수 없다면 :김초엽 소설,지은이: 김초엽,9791190090018,496g
1,2,2,달러구트 꿈 백화점.이미예 장편소설,지은이: 이미예,9791165341909,358g
2,3,3,지구에서 한아뿐 :정세랑 장편소설,지은이: 정세랑,9791188862290,296g
3,4,4,"시선으로부터, :정세랑 장편소설",지은이: 정세랑,9788954672214,412g
4,5,5,아몬드 :손원평 장편소설,지은이: 손원평,9788936434267,388g
5,6,6,피프티 피플 :정세랑 장편소설,지은이: 정세랑,9788936434243,512g
6,7,7,목소리를 드릴게요 :정세랑 소설집,지은이: 정세랑,9791165300005,318g
7,8,8,나미야 잡화점의 기적 :히가시노 게이고 장편소설,지은이: 히가시노 게이고 ;옮긴이: 양윤옥,9788972756194,
8,9,9,선량한 차별주의자,김지혜 지음,9788936477196,324g
9,10,9,쇼코의 미소 :최은영 소설,지은이: 최은영,9788954641630,406g
